# Introduction to Pandas DataFrame

This notebook provides an introduction to working with Pandas DataFrames, a fundamental tool for data manipulation and analysis in Python.

## What you'll learn
- Creating and manipulating DataFrames
- Loading data from various sources
- Basic data cleaning and transformation
- Data visualization with Pandas
- Integration with machine learning workflows

In [ ]:
# Install pandas if not already installed
!pip install pandas matplotlib seaborn

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options
pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', 10)

# Set plotting style
sns.set(style="whitegrid")

## Creating DataFrames

There are multiple ways to create a DataFrame in pandas:

In [ ]:
# Create a DataFrame from a dictionary
data = {
    'Name': ['Alice', 'Bob', 'Charlie', 'David', 'Eva'],
    'Age': [25, 30, 35, 40, 45],
    'City': ['New York', 'San Francisco', 'Los Angeles', 'Chicago', 'Boston'],
    'Salary': [50000, 60000, 70000, 80000, 90000]
}

df = pd.DataFrame(data)
print("DataFrame from dictionary:")
df

In [ ]:
# Create a DataFrame from lists
names = ['Alice', 'Bob', 'Charlie', 'David', 'Eva']
ages = [25, 30, 35, 40, 45]
cities = ['New York', 'San Francisco', 'Los Angeles', 'Chicago', 'Boston']
salaries = [50000, 60000, 70000, 80000, 90000]

df2 = pd.DataFrame(list(zip(names, ages, cities, salaries)),
                   columns=['Name', 'Age', 'City', 'Salary'])
print("\nDataFrame from lists:")
df2

## Loading Data from External Sources

Pandas can load data from various file formats:

In [ ]:
# Download a sample CSV file
!wget -q https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv

# Load CSV file
titanic_df = pd.read_csv('titanic.csv')
print("Titanic dataset:")
titanic_df.head()

## Basic DataFrame Operations

Let's explore some basic operations on DataFrames:

In [ ]:
# Get basic information about the DataFrame
print("DataFrame shape:", titanic_df.shape)
print("\nDataFrame columns:", titanic_df.columns.tolist())
print("\nDataFrame data types:")
titanic_df.dtypes

In [ ]:
# Get summary statistics
titanic_df.describe()

## Data Cleaning and Transformation

Let's perform some basic data cleaning operations:

In [ ]:
# Check for missing values
print("Missing values per column:")
titanic_df.isnull().sum()

In [ ]:
# Fill missing values
# Fill Age with median
median_age = titanic_df['Age'].median()
titanic_df['Age'].fillna(median_age, inplace=True)

# Fill Cabin with 'Unknown'
titanic_df['Cabin'].fillna('Unknown', inplace=True)

# Fill Embarked with most common value
most_common_embarked = titanic_df['Embarked'].mode()[0]
titanic_df['Embarked'].fillna(most_common_embarked, inplace=True)

# Check missing values again
print("Missing values after filling:")
titanic_df.isnull().sum()

In [ ]:
# Create new features
# Extract title from Name
titanic_df['Title'] = titanic_df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# Create family size feature
titanic_df['FamilySize'] = titanic_df['SibSp'] + titanic_df['Parch'] + 1

# Create age groups
titanic_df['AgeGroup'] = pd.cut(titanic_df['Age'], bins=[0, 12, 18, 65, 100], 
                               labels=['Child', 'Teenager', 'Adult', 'Elderly'])

# Display the new features
titanic_df[['Name', 'Title', 'SibSp', 'Parch', 'FamilySize', 'Age', 'AgeGroup']].head()

## Data Visualization with Pandas

Pandas provides built-in plotting capabilities based on matplotlib:

In [ ]:
# Survival rate by passenger class
survival_by_class = titanic_df.groupby('Pclass')['Survived'].mean()

plt.figure(figsize=(10, 6))
survival_by_class.plot(kind='bar', color='skyblue')
plt.title('Survival Rate by Passenger Class')
plt.xlabel('Passenger Class')
plt.ylabel('Survival Rate')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Age distribution by survival
plt.figure(figsize=(12, 6))
sns.histplot(data=titanic_df, x='Age', hue='Survived', bins=30, kde=True, element='step')
plt.title('Age Distribution by Survival')
plt.xlabel('Age')
plt.ylabel('Count')
plt.legend(['Not Survived', 'Survived'])
plt.grid(linestyle='--', alpha=0.7)
plt.show()

## Integration with Machine Learning Workflows

Pandas DataFrames integrate seamlessly with scikit-learn for machine learning:

In [ ]:
# Install scikit-learn if not already installed
!pip install scikit-learn

In [ ]:
# Prepare data for machine learning
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Select features and target
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
X = titanic_df[features].copy()
y = titanic_df['Survived']

# Encode categorical variables
label_encoders = {}
for column in ['Sex', 'Embarked']:
    le = LabelEncoder()
    X[column] = le.fit_transform(X[column])
    label_encoders[column] = le

# Scale numerical features
scaler = StandardScaler()
X[['Age', 'Fare']] = scaler.fit_transform(X[['Age', 'Fare']])

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)
X_train.head()

In [ ]:
# Train a simple model
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Create and train the model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Model accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Visualize feature importance
feature_importance = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importances_
})
feature_importance = feature_importance.sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance, palette='viridis')
plt.title('Feature Importance')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## Conclusion

In this notebook, we've covered the basics of working with Pandas DataFrames:

1. Creating DataFrames from different data sources
2. Loading data from external files
3. Basic data exploration and manipulation
4. Data cleaning and feature engineering
5. Data visualization
6. Integration with machine learning workflows

Pandas is an essential tool for data analysis and preprocessing in machine learning projects. It provides a flexible and powerful interface for working with structured data, making it easier to prepare your data for modeling.

## Next Steps

To continue learning about data manipulation and analysis with Pandas, you might want to explore:

1. Advanced indexing and selection techniques
2. GroupBy operations for aggregation and transformation
3. Time series analysis with Pandas
4. Handling large datasets efficiently
5. Integration with other data processing libraries like Dask or RAPIDS cuDF

Check out the [Pandas documentation](https://pandas.pydata.org/docs/) for more information and tutorials.